# Word Embeddings: TF-IDF and Word2Vec

This notebook covers two fundamental approaches to text representation:
1. **TF-IDF** -- sparse, frequency-based vectors
2. **Word2Vec** -- dense, learned embeddings that capture semantic similarity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

try:
    from gensim.models import Word2Vec
    HAS_GENSIM = True
except ImportError:
    HAS_GENSIM = False
    print('gensim not installed -- pip install gensim')

%matplotlib inline

In [ ]:
# Corpus of sentences
corpus = [
    "machine learning algorithms require large datasets for training",
    "deep learning is a subset of machine learning using neural networks",
    "natural language processing deals with text and speech data",
    "word embeddings represent words as dense numerical vectors",
    "transformer models have revolutionised natural language processing",
    "supervised learning uses labelled data to train predictive models",
    "unsupervised learning discovers hidden patterns without labels",
    "reinforcement learning trains agents through rewards and penalties",
    "topological data analysis studies the shape of data",
    "persistent homology is a key tool in topological data analysis"
]
print(f"Corpus: {len(corpus)} documents")

## 1. TF-IDF (Term Frequency -- Inverse Document Frequency)

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}$$

- Words frequent in a document but rare across the corpus get high weight.
- Common words ("the", "is") get low weight.

In [ ]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

print(f"TF-IDF matrix shape: {X_tfidf.shape}  (documents x vocabulary)")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

# Top TF-IDF terms for the first document
feature_names = tfidf.get_feature_names_out()
scores = X_tfidf[0].toarray().flatten()
top_idx = scores.argsort()[::-1][:5]
print(f"\nTop terms in doc 0: {[(feature_names[i], f'{scores[i]:.3f}') for i in top_idx]}")

In [ ]:
# Document similarity using TF-IDF cosine similarity
sim_matrix = cosine_similarity(X_tfidf)

plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, cmap='Blues')
plt.colorbar(label='Cosine similarity')
plt.title('Document Similarity (TF-IDF)')
plt.xlabel('Document')
plt.ylabel('Document')
plt.tight_layout()
plt.show()

# Most similar pair
np.fill_diagonal(sim_matrix, 0)
i, j = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
print(f"Most similar docs: {i} and {j} (sim={sim_matrix[i,j]:.3f})")
print(f"  Doc {i}: {corpus[i]}")
print(f"  Doc {j}: {corpus[j]}")

## 2. Word2Vec

Word2Vec (Mikolov et al., 2013) learns dense embeddings by predicting context:
- **CBOW**: predict target word from context
- **Skip-gram**: predict context from target word

The resulting vectors capture semantic relationships: $\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$

In [ ]:
if HAS_GENSIM:
    # Tokenise corpus
    tokenized = [sentence.split() for sentence in corpus]
    
    # Train Word2Vec (small corpus, so use small dimensions)
    w2v = Word2Vec(tokenized, vector_size=50, window=5, min_count=1,
                   sg=1, epochs=100, seed=42)  # sg=1 for skip-gram
    
    print(f"Vocabulary size: {len(w2v.wv)}")
    print(f"Embedding dimension: {w2v.wv.vector_size}")
    
    # Most similar words to 'learning'
    print(f"\nMost similar to 'learning':")
    for word, score in w2v.wv.most_similar('learning', topn=5):
        print(f"  {word:<20} {score:.3f}")

In [ ]:
if HAS_GENSIM:
    # Visualise embeddings in 2D
    words = list(w2v.wv.key_to_index.keys())
    vectors = np.array([w2v.wv[w] for w in words])
    
    pca = PCA(n_components=2)
    coords = pca.fit_transform(vectors)
    
    plt.figure(figsize=(12, 8))
    plt.scatter(coords[:, 0], coords[:, 1], s=20, alpha=0.5)
    # Label selected words
    highlight = ['learning', 'data', 'models', 'topological', 'language',
                 'neural', 'networks', 'analysis', 'training', 'embeddings']
    for word in highlight:
        if word in words:
            idx = words.index(word)
            plt.annotate(word, (coords[idx, 0], coords[idx, 1]),
                        fontsize=11, fontweight='bold', color='red')
    plt.title('Word2Vec Embeddings (PCA projection)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
if HAS_GENSIM:
    # Word similarity matrix for selected terms
    selected = ['learning', 'data', 'analysis', 'language', 'models', 'topological']
    selected = [w for w in selected if w in w2v.wv]
    sim = np.array([[w2v.wv.similarity(w1, w2) for w2 in selected] for w1 in selected])
    
    plt.figure(figsize=(7, 5))
    plt.imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1)
    plt.xticks(range(len(selected)), selected, rotation=45)
    plt.yticks(range(len(selected)), selected)
    plt.colorbar(label='Cosine similarity')
    plt.title('Word2Vec Similarity Matrix')
    plt.tight_layout()
    plt.show()

## Key Takeaways

| Method | Type | Pros | Cons |
|--------|------|------|------|
| TF-IDF | Sparse | Simple, interpretable | No semantics, high-dimensional |
| Word2Vec | Dense | Captures semantics | Needs large corpus, static embeddings |

Modern approaches use **contextual embeddings** (BERT, GPT) where the same word gets different vectors depending on context.

**Next:** Text classification (sentiment analysis).